# Neural Network By Hand!

**Deep Learning Indaba 2026 — Lagos, Nigeria**

**Session Leaders:** Pelumi Victor, Catherine Essuman & Claire David

---

In this exercise, you will code a neural network **from scratch** — no frameworks, just NumPy and the math you learned in the tutorial. Don't worry, it's extremely guided. The two most important points being that **you learn** and **you have fun**.

You will build a network to solve the classic **XOR problem** — a dataset that is *not* linearly separable, proving that a simple perceptron won't cut it.

**Outline:**
- Part 0: The Math (warm-up)
- Part I: Code Your Neural Network!
  - 1.1 Get the Data
  - 1.2 Functions (weighted sum, activations, cost, derivatives)
  - 1.3 Feedforward Propagation
  - 1.4 Training (forward + backward + gradient descent)
  - 1.5 Visualize Results
- Part II: Generalize Your Network (bonus)
- Part III: Double Spiral Challenge (bonus)

**Adapted from:** [Claire David's T5 — Neural Network by Hand!](https://clairedavid.github.io/intro_to_ml/tutorials/t05_nn_by_hand.html)

---

### How to use this notebook

Look for **`#...`** — that's where you write your code. Everything else is provided for you. Read the instructions carefully and refer to the equations!

---
## 0. Setup

Run this cell first — it installs dependencies and defines helper functions you'll need later.

In [ ]:
# Install dependencies
!pip install numpy pandas matplotlib gdown --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

# ---------- Plot settings ----------
FONTSIZE = 16
params = {
    'figure.figsize' : (6, 6),
    'axes.labelsize' : FONTSIZE,
    'axes.titlesize' : FONTSIZE + 2,
    'legend.fontsize': FONTSIZE,
    'xtick.labelsize': FONTSIZE,
    'ytick.labelsize': FONTSIZE,
    'xtick.color'    : 'black',
    'ytick.color'    : 'black',
    'axes.facecolor' : 'white',
    'axes.edgecolor' : 'black',
    'axes.titlepad'  :  20,
    'axes.labelpad'  :  10,
}
plt.rcParams.update(params)

XNAME = 'x1'; XLABEL = r'$x_1$'
YNAME = 'x2'; YLABEL = r'$x_2$'
RANGE = (-6, 6); STEP = 0.1

In [ ]:
# ---------- Helper functions (provided — do not modify) ----------

def predict(output_node, boundary_value):
    """Convert continuous output to binary predictions."""
    output_node = output_node.reshape(-1, 1, 1)
    predictions = (output_node > boundary_value).astype(int)
    return predictions


def plot_cost_vs_iter(train_costs, test_costs, title="Cost evolution"):
    """Plot training and test cost over iterations."""
    fig, ax = plt.subplots(figsize=(8, 6))
    iters = np.arange(1, len(train_costs) + 1)
    ax.plot(iters, train_costs, color='red', lw=1, label='Training set')
    ax.plot(iters, test_costs, color='blue', lw=1, label='Testing set')
    ax.set_xlabel("Number of iterations")
    ax.set_xlim(1, iters[-1])
    ax.set_ylabel("Cost")
    ax.legend(loc="upper right", frameon=False)
    ax.set_title(title)
    plt.show()


def get_decision_surface(weights, biases, boundary=0.5, range=RANGE, step=STEP):
    """Compute decision surface over a grid of points."""
    x1v, x2v = np.meshgrid(
        np.arange(range[0], range[1] + step, step),
        np.arange(range[0], range[1] + step, step),
    )
    X_grid = np.c_[x1v.ravel(), x2v.ravel()].reshape(-1, 2)

    output = feedforward(X_grid, weights, biases)[-1]  # output node
    Ypred_grid = predict(output, boundary)

    return (x1v, x2v, Ypred_grid.reshape(x1v.shape))


def plot_scatter(sig, bkg, ds=None,
                 xname=XNAME, xlabel=XLABEL,
                 yname=YNAME, ylabel=YLABEL,
                 range=RANGE, step=STEP, title="Scatter plot"):
    """Scatter plot with optional decision surface."""
    fig, ax = plt.subplots()

    if ds:
        (xx, yy, Z) = ds
        ax.contourf(xx, yy, Z, levels=[0, 0.5, 1],
                    colors=['orange', 'dodgerblue'], alpha=0.3)

    ax.scatter(sig[xname], sig[yname], marker='o', s=10,
               c='dodgerblue', alpha=1, label='Positive class')
    ax.scatter(bkg[xname], bkg[yname], marker='o', s=10,
               c='orange', alpha=1, label='Negative class')

    ax.set_xlim(range); ax.set_xlabel(xlabel)
    ax.set_ylim(range); ax.set_ylabel(ylabel)
    ax.legend(bbox_to_anchor=(1.04, 0.5), loc="center left", frameon=False)
    ax.set_title(title)
    plt.show()


def print_every(iter_idx):
    """Decide whether to print at this iteration (for cleaner output)."""
    if iter_idx <= 100:
        return iter_idx % 10 == 0
    elif iter_idx <= 1000:
        return iter_idx % 100 == 0
    else:
        return iter_idx % 1000 == 0


print("Setup complete!")

---
## Part 0: The Math (warm-up)

Before coding, let's make sure the key equations are clear. Our network has **2 hidden layers** with **tanh** activation and a **sigmoid** output:

**Feedforward (general rule for layer $\ell$):**

$$z^{(i,\ell)} = (W^{(\ell)})^T \cdot a^{(i,\ell-1)} + b^{(\ell)}$$
$$a^{(i,\ell)} = f(z^{(i,\ell)})$$

**Backpropagation — error at last layer $L$:**

$$\delta^{(i,L)} = f'(z^{(i,L)}) \odot \mathcal{L}'(a^{(i,L)}, y^{(i)})$$

**Recursive error for earlier layers:**

$$\delta^{(i,\ell)} = f'(z^{(i,\ell)}) \odot \left[ W^{(\ell+1)} \cdot \delta^{(i,\ell+1)} \right]$$

**Cost gradients:**

$$\frac{\partial J}{\partial W^{(\ell)}} = \frac{1}{m} \sum_i a^{(i,\ell-1)} \cdot (\delta^{(i,\ell)})^T$$

$$\frac{\partial J}{\partial b^{(\ell)}} = \frac{1}{m} \sum_i \delta^{(i,\ell)}$$

**Update rules:**

$$W^{(\ell)} \leftarrow W^{(\ell)} - \alpha \cdot \frac{\partial J}{\partial W^{(\ell)}}$$
$$b^{(\ell)} \leftarrow b^{(\ell)} - \alpha \cdot \frac{\partial J}{\partial b^{(\ell)}}$$

where $\odot$ is element-wise multiplication and $\alpha$ is the learning rate.

**Quick reference:**
- Matrix multiplication in Python: `A @ B`
- Element-wise multiplication: `A * B`
- Transpose of matrix `M`: `M.T`
- Transpose 3D array: `np.transpose(arr, (0, 2, 1))`
- Sum over samples (axis 0): `np.sum(arr, axis=0)`

---
## Part I: Code Your Neural Network!

### 1.1 Get the Data

We'll use an XOR-like dataset — two input features ($x_1$, $x_2$) with a binary label. This is the classic problem that a single perceptron **cannot** solve.

#### 1.1.1 Download the data

In [ ]:
import gdown

# Training data
gdown.download(
    'https://drive.google.com/uc?export=download&id=1oyWJCdJ7LrHN_1xUMkiED8FR-TV8qlFn',
    'ml_tutorial_5_data_train.csv', quiet=False
)

# Test data
gdown.download(
    'https://drive.google.com/uc?export=download&id=1g7DVcMQMGfce2fQ8x3FeZqXHbdrrcbQP',
    'ml_tutorial_5_data_test.csv', quiet=False
)

df_train = pd.read_csv('ml_tutorial_5_data_train.csv')
df_test  = pd.read_csv('ml_tutorial_5_data_test.csv')

print(f"Training samples: {len(df_train)}")
print(f"Test samples:     {len(df_test)}")
df_train.head()

#### 1.1.2 Split signal vs background

Create `sig` and `bkg` dataframes by filtering on the target column. We'll use these for plotting.

In [ ]:
# Identify the target column name
target_col = [c for c in df_train.columns if c not in ['x1', 'x2']][0]
print(f"Target column: '{target_col}'")

sig = df_train[df_train[target_col] > 0.5]
bkg = df_train[df_train[target_col] < 0.5]

print(f"Signal samples:     {len(sig)}")
print(f"Background samples: {len(bkg)}")

# Visualize the XOR dataset
plot_scatter(sig, bkg, title="XOR Dataset (not linearly separable!)")

#### 1.1.3 DataFrame to NumPy

Convert the data to NumPy arrays for our computations.

In [ ]:
inputs = ['x1', 'x2']

X_train = df_train[inputs].values
y_train = df_train[target_col].values

X_test = df_test[inputs].values
y_test = df_test[target_col].values

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_test shape:  {y_test.shape}")

### 1.2 Functions

Now we build the individual pieces we need: weighted sum, activation functions, cost function, and their derivatives.

#### 1.2.1 Weighted Sum

Create the function `z` that computes the weighted sum:

$$z = W^T \cdot a + b$$

Use the `@` operator for matrix multiplication.

In [ ]:
def z(a_prev, W, b):
    """Compute the weighted sum: z = W^T * a_prev + b"""
    #...
    return #...

#### 1.2.2 Activation Functions and Derivatives

We need two activation functions and their derivatives:

**Sigmoid:** $\;\sigma(z) = \frac{1}{1 + e^{-z}}$, $\quad\sigma'(z) = \sigma(z)(1 - \sigma(z))$

**Tanh:** $\;\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}$, $\quad\tanh'(z) = 1 - \tanh^2(z)$

In [ ]:
def sigmoid(z):
    return #...

def tanh(z):
    return #...

def sigmoid_prime(z):
    """Derivative of sigmoid."""
    return #...

def tanh_prime(z):
    """Derivative of tanh."""
    return #...

In [ ]:
# Quick check — run this to verify your functions
test_z = np.array([0.0, 1.0, -1.0])
print(f"sigmoid([0, 1, -1])       = {sigmoid(test_z)}")
print(f"  expected:               ≈ [0.5, 0.731, 0.269]")
print(f"sigmoid_prime([0, 1, -1]) = {sigmoid_prime(test_z)}")
print(f"  expected:               ≈ [0.25, 0.197, 0.197]")
print(f"tanh([0, 1, -1])          = {tanh(test_z)}")
print(f"  expected:               ≈ [0.0, 0.762, -0.762]")
print(f"tanh_prime([0, 1, -1])    = {tanh_prime(test_z)}")
print(f"  expected:               ≈ [1.0, 0.420, 0.420]")

#### 1.2.3 Cross-Entropy Cost Function

The binary cross-entropy cost averages the loss over all $m$ samples:

$$J = -\frac{1}{m} \sum_{i=1}^{m} \left[ y^{(i)} \log(\hat{y}^{(i)}) + (1 - y^{(i)}) \log(1 - \hat{y}^{(i)}) \right]$$

**Hint:** Use a small epsilon (e.g. `1e-8`) inside the `log` to avoid `log(0)`.

In [ ]:
def cross_entropy_cost(y_pred, y_true):
    """Binary cross-entropy cost function."""
    #...
    return #...

#### 1.2.4 Derivative of the Loss

The derivative of the binary cross-entropy loss with respect to $\hat{y}$:

$$\frac{\partial \mathcal{L}}{\partial \hat{y}^{(i)}} = -\frac{y^{(i)}}{\hat{y}^{(i)}} + \frac{1 - y^{(i)}}{1 - \hat{y}^{(i)}}$$

**Exercise:** First, derive this on paper from $\mathcal{L} = -[y\log(\hat{y}) + (1-y)\log(1-\hat{y})]$. Then code it:

In [ ]:
def L_prime(y_pred, y_true):
    """Derivative of binary cross-entropy loss w.r.t. y_pred."""
    #...
    return #...

### 1.3 Feedforward Propagation

#### 1.3.1 Implementing the Feedforward Function

Our network architecture:
- **Input**: 2 features ($x_1$, $x_2$)
- **Hidden layer 1**: `q` nodes, **tanh** activation
- **Hidden layer 2**: `r` nodes, **tanh** activation
- **Output layer**: 1 node, **sigmoid** activation

The weights and biases are passed as lists. The input data is reshaped to 3D: `(m, n_features, 1)` for proper matrix operations.

Complete the function below. Use your `z()` function for the weighted sum, then apply the appropriate activation.

**Hint:** For the matrix multiplication with 3D arrays, `W.T @ a` will broadcast correctly since `a` has shape `(m, j, 1)` and `W` has shape `(j_prev, j)`.

In [ ]:
def feedforward(input_X, weights, biases):
    """
    Forward pass through a 2-hidden-layer network.

    Args:
        input_X: input data, shape (m, 2)
        weights: list [W1, W2, W3]
        biases:  list [b1, b2, b3]

    Returns:
        nodes: list [a0, z1, a1, z2, a2, z3, a3]
    """
    W1, W2, W3 = weights
    b1, b2, b3 = biases

    m  = len(input_X)
    a0 = input_X.reshape((m, -1, 1))  # shape: (m, 2, 1)

    # First hidden layer (tanh)
    z1 = #...
    a1 = #...

    # Second hidden layer (tanh)
    z2 = #...
    a2 = #...

    # Output layer (sigmoid)
    z3 = #...
    a3 = #...

    nodes = [a0, z1, a1, z2, a2, z3, a3]
    return nodes

#### 1.3.2 Predict

The `predict` function is already provided in the setup. Think about:
- What is the `output_node` in our network? *(Hint: it's the last element returned by `feedforward`)*
- What does `predict` return? *(Binary: 0 or 1)*
- How would you call it after `feedforward`?

### 1.4 Neural Network Training

Time to train! Complete the training loop below. The hyperparameters are given.

**Weight initialization:** Use `np.random.random((rows, cols))` — note the double parentheses. Think about what dimensions each weight matrix needs.

**Backpropagation reminders:**

| Error | Formula |
|:--|:--|
| $\delta_3$ (output) | $f'(z_3) \odot \mathcal{L}'(a_3, y)$ |
| $\delta_2$ | $f'(z_2) \odot (W_3 \cdot \delta_3)$ |
| $\delta_1$ | $f'(z_1) \odot (W_2 \cdot \delta_2)$ |

| Gradient | Formula |
|:--|:--|
| $\frac{\partial J}{\partial W_\ell}$ | $\frac{1}{m} \sum_i a_{\ell-1} \cdot \delta_\ell^T$ |
| $\frac{\partial J}{\partial b_\ell}$ | $\frac{1}{m} \sum_i \delta_\ell$ |

**NumPy tips:**
- `@` for matrix multiplication, `*` for element-wise
- Transpose 3D: `np.transpose(arr, (0, 2, 1))`
- Sum over samples: `np.sum(arr, axis=0)`

In [ ]:
# =====================
#   HYPERPARAMETERS
# =====================
alpha  = 0.05   # learning rate
n_iter = 5000   # number of iterations

# =====================
#   INITIALIZATION
# =====================
m = len(X_train)       # number of training samples
n = X_train.shape[1]   # number of input features (2)
q = 3                  # nodes in first hidden layer
r = 2                  # nodes in second hidden layer

# Weight matrices: W_l has shape (n_prev, n_curr)
# Bias vectors:    b_l has shape (n_curr, 1)
W1 = #...  # shape (n, q) = (2, 3)
W2 = #...  # shape (q, r) = (3, 2)
W3 = #...  # shape (r, 1) = (2, 1)

b1 = #...  # shape (q, 1) = (3, 1)
b2 = #...  # shape (r, 1) = (2, 1)
b3 = #...  # shape (1, 1)

# Reshape targets to 3D for consistent operations
y_train_3d = np.reshape(y_train, (-1, 1, 1))
y_test_3d  = np.reshape(y_test,  (-1, 1, 1))

# Storage for cost curves
costs_train = []
costs_test  = []

# Set to True to print shape info for debugging
debug = False

print("Starting the training\n")

# ===========================
#   START TRAINING LOOP
# ===========================
for iter_idx in range(1, n_iter + 1):

    # ----- FORWARD PROPAGATION -----

    # Feedforward on TEST data (for monitoring, no backprop):
    nodes_test = #...
    ypred_test = #...  # last element of nodes_test

    # Feedforward on TRAINING data:
    a0, z1, a1, z2, a2, z3, a3 = #...
    ypred_train = #...  # same as a3

    # Cost computation
    cost_train = cross_entropy_cost(ypred_train, y_train_3d)
    cost_test  = cross_entropy_cost(ypred_test,  y_test_3d)
    costs_train.append(cost_train)
    costs_test.append(cost_test)

    # ----- BACKWARD PROPAGATION -----

    # Errors (deltas):
    delta_3 = #...   # sigmoid_prime(z3) * L_prime(a3, y_train_3d)
    delta_2 = #...   # tanh_prime(z2) * (W3 @ delta_3)
    delta_1 = #...   # tanh_prime(z1) * (W2 @ delta_2)

    # Partial derivatives (cost gradients):
    # dCost/dW = (1/m) * sum_i [ a_prev * delta^T ]
    # Use np.transpose(delta, (0,2,1)) to get delta^T for 3D arrays
    dCostdW3 = #...
    dCostdW2 = #...
    dCostdW1 = #...

    # dCost/db = (1/m) * sum_i [ delta ]
    dCostdb3 = #...
    dCostdb2 = #...
    dCostdb1 = #...

    # Print progress
    if print_every(iter_idx):
        print(
            f"Iteration {iter_idx:>5d}\t"
            f"Train cost: {cost_train:.5f}\t"
            f"Test cost: {cost_test:.5f}\t"
            f"Diff: {cost_test - cost_train:.2e}"
        )
    if debug and iter_idx < 3:
        print(
            f"  Shapes -> a0:{a0.shape} a1:{a1.shape} a2:{a2.shape} a3:{a3.shape} | "
            f"W1:{W1.shape} W2:{W2.shape} W3:{W3.shape} | "
            f"dW1:{dCostdW1.shape} dW2:{dCostdW2.shape} dW3:{dCostdW3.shape}"
        )

    # ----- GRADIENT DESCENT UPDATE -----
    W3 = #...   # W3 - alpha * dCostdW3
    W2 = #...
    W1 = #...
    b3 = #...   # b3 - alpha * dCostdb3
    b2 = #...
    b1 = #...

print(f"\nEnd of gradient descent after {iter_idx} iterations")

### 1.5 Visualize Results

#### 1.5.1 Cost Evolution

Plot the training and test cost over iterations. Both should decrease — if test cost diverges from train cost, you might be overfitting.

In [ ]:
plot_cost_vs_iter(costs_train, costs_test, title="Cost evolution")

#### 1.5.2 Decision Boundary

Use the provided `get_decision_surface` and `plot_scatter` functions to see if your network learned the XOR pattern.

If all goes well, you should see four distinct regions!

In [ ]:
weights = [W1, W2, W3]
biases  = [b1, b2, b3]

ds = get_decision_surface(weights, biases, boundary=0.5)
plot_scatter(sig, bkg, ds=ds, title="Neural network decision boundary")

In [ ]:
# Final accuracy on test set
nodes_final = feedforward(X_test, weights, biases)
test_preds = predict(nodes_final[-1], 0.5).flatten()
test_acc = np.mean(test_preds == y_test)
print(f"Test accuracy: {test_acc:.1%}")

### You just coded a neural network by hand!

If your decision boundary correctly separates the four XOR clusters, congratulations! You understand forward propagation, backpropagation, and gradient descent at the deepest level.

---

## Part II: Generalize Your Network (Bonus)

We hardcoded two hidden layers above. Can you make it work for **any depth**?

### 2.1 Dynamic Depth

Modify your code so that the architecture is defined by an array. For example:

```python
layers = [3, 3, 2, 1]
```

would create a network with 3 nodes in the first hidden layer, 3 in the second, 2 in the third, and 1 output.

**Hint:** Use loops over the layers instead of hardcoded `W1`, `W2`, `W3`.

In [ ]:
# YOUR CODE HERE: Generalize feedforward and backprop to any depth


### 2.2 Multiple Output Nodes

Extend your code further to support more than one output node. You'll need to switch from binary cross-entropy to **categorical cross-entropy**.

Try it on a multi-class dataset from scikit-learn: [sklearn.datasets](https://scikit-learn.org/stable/api/sklearn.datasets.html)

In [ ]:
# YOUR CODE HERE: Multi-class extension


---
## Part III: Double Spiral Challenge (Bonus)

The double spiral is a much harder dataset. Can your network learn it?

**Hints:**
- You'll need more layers and more nodes
- Try implementing **momentum** in your update rule
- Experiment with different learning rates

In [ ]:
def generate_double_spiral(n_points_per_class=500, noise=0.2, test_fraction=0.2, seed=42):
    """
    Generate a 2D double spiral dataset.

    Returns:
        X_train, y_train, X_test, y_test
    """
    np.random.seed(seed)

    theta = np.linspace(0, 4 * np.pi, n_points_per_class)

    # Spiral 1 (label 0)
    r1 = theta
    x1 = r1 * np.cos(theta) + noise * np.random.randn(n_points_per_class)
    x2 = r1 * np.sin(theta) + noise * np.random.randn(n_points_per_class)
    y1 = np.zeros((n_points_per_class, 1))

    # Spiral 2 (label 1), shifted by pi
    r2 = theta
    x1_2 = r2 * np.cos(theta + np.pi) + noise * np.random.randn(n_points_per_class)
    x2_2 = r2 * np.sin(theta + np.pi) + noise * np.random.randn(n_points_per_class)
    y2 = np.ones((n_points_per_class, 1))

    # Stack and shuffle
    X = np.vstack((np.column_stack((x1, x2)), np.column_stack((x1_2, x2_2))))
    y = np.vstack((y1, y2))
    idx = np.arange(len(X))
    np.random.shuffle(idx)
    X, y = X[idx], y[idx]

    # Train/test split
    n_test = int(len(X) * test_fraction)
    return X[n_test:], y[n_test:], X[:n_test], y[:n_test]


def plot_spiral(X_train, y_train, X_test=None, y_test=None, ds=None, title="Double Spiral"):
    """Plot the double spiral dataset with optional decision surface."""
    fig, ax = plt.subplots(figsize=(6, 6))

    if ds:
        (xx, yy, Z) = ds
        ax.contourf(xx, yy, Z, levels=[0, 0.5, 1],
                    colors=['orange', 'dodgerblue'], alpha=0.3)

    ax.scatter(X_train[:, 0][y_train[:, 0] == 1],
               X_train[:, 1][y_train[:, 0] == 1],
               marker='o', s=10, c='dodgerblue', alpha=1, label="Positive (train)")
    ax.scatter(X_train[:, 0][y_train[:, 0] == 0],
               X_train[:, 1][y_train[:, 0] == 0],
               marker='o', s=10, c='orange', alpha=1, label="Negative (train)")

    if X_test is not None and y_test is not None:
        ax.scatter(X_test[:, 0][y_test[:, 0] == 1],
                   X_test[:, 1][y_test[:, 0] == 1],
                   marker='x', s=25, c='dodgerblue', alpha=0.8, label="Positive (test)")
        ax.scatter(X_test[:, 0][y_test[:, 0] == 0],
                   X_test[:, 1][y_test[:, 0] == 0],
                   marker='x', s=25, c='orange', alpha=0.8, label="Negative (test)")

    ax.set_xlim(-15, 15); ax.set_ylim(-15, 15)
    ax.set_xlabel(r'$x_1$'); ax.set_ylabel(r'$x_2$')
    ax.set_title(title)
    ax.legend(bbox_to_anchor=(1.04, 0.5), loc="center left", frameon=False)
    plt.show()


# Generate and visualize
X_sp_train, y_sp_train, X_sp_test, y_sp_test = generate_double_spiral()
plot_spiral(X_sp_train, y_sp_train, X_sp_test, y_sp_test)

In [ ]:
# YOUR CODE HERE: Train a network to solve the double spiral!
# You'll likely need:
#   - More hidden layers and nodes (try [2, 32, 32, 16, 1])
#   - More iterations (10000+)
#   - A good learning rate
#   - Possibly momentum in the update rule


---
## Well done!

You've built a neural network from the ground up. Every time you call `model.fit()` or `loss.backward()` in a framework, this is what's happening under the hood.

### Resources
- [Claire David's full course](https://clairedavid.github.io/intro_to_ml/nn/nn_0_intro.html)
- [Original T5 tutorial](https://clairedavid.github.io/intro_to_ml/tutorials/t05_nn_by_hand.html)
- [TensorFlow Playground](https://playground.tensorflow.org)
- [3Blue1Brown — Neural Networks](https://www.youtube.com/playlist?list=PLZHQObOWTQDNU6R1_67000Dx_ZCJB-3pi)